In [2]:
from google.colab import drive

In [3]:
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import os
os.makedirs('/content/drive/MyDrive/task3_churn', exist_ok=True)
%cd /content/drive/MyDrive/task3_churn
!pwd

/content/drive/MyDrive/task3_churn
/content/drive/MyDrive/task3_churn


In [5]:
import pandas as pd, numpy as np
np.random.seed(7)

n, today = 200, pd.Timestamp("2026-10-19")
df = pd.DataFrame({
    "customer_id": [f"C{1000+i}" for i in range(n)],
    "last_order_date": today - pd.to_timedelta(np.random.exponential(25, n).astype(int), unit="D"),
    "orders_prev_30d": np.random.randint(1, 10, n),
    "total_spend": np.random.randint(100, 5000, n),
})
df["orders_last_30d"] = (df["orders_prev_30d"] * np.random.uniform(0, 1.4, n)).round().astype(int)


In [ ]:
inactive = (today - df["last_order_date"]).dt.days > 30
df.loc[inactive, "orders_last_30d"] = 0


In [6]:
df.to_csv("customers.csv", index=False)
print("customers.csv created")
df.head()

customers.csv created


,customer_id,last_order_date,orders_prev_30d,total_spend,orders_last_30d
0,C1000,2026-10-18,7,3964,2
1,C1001,2026-09-12,1,1739,1
2,C1002,2026-10-05,7,926,9
3,C1003,2026-09-17,5,1928,1
4,C1004,2026-07-16,7,4750,8


In [8]:
import pandas as pd

today = pd.Timestamp("2026-10-19")
df = pd.read_csv("customers.csv", parse_dates=["last_order_date"])


In [7]:
df["days_inactive"] = (today - df["last_order_date"]).dt.days
df["sig_inactive"] = df["days_inactive"] >= 30


In [9]:
df["sig_freq_drop"] = df["orders_last_30d"] <= df["orders_prev_30d"] * 0.5


In [13]:
df["days_inactive"] = (today - df["last_order_date"]).dt.days
df["sig_inactive"] = df["days_inactive"] >= 30
df["sig_freq_drop"] = df["orders_last_30d"] <= df["orders_prev_30d" ] * 0.5

high_value = df["total_spend"] >= df["total_spend"].quantile(0.75)
df["sig_high_value_quiet"] = high_value & (df["days_inactive"] >= 20)

signals = ["sig_inactive", "sig_freq_drop", "sig_high_value_quiet"]
df["risk_score"] = df[signals].sum(axis=1)

at_risk = (df[df["risk_score"] >= 1]
           .sort_values(["risk_score", "total_spend"], ascending=False)
           [["customer_id", "risk_score", "days_inactive", "orders_prev_30d",
             "orders_last_30d", "total_spend"] + signals])

at_risk.to_csv("at_risk_customers.csv", index=False)

print(f"{len(at_risk)} of {len(df)} customers flagged")
print(f"{(df['risk_score'] == 3).sum()} customers have a risk score of 3\n")
at_risk.head(15)

123 of 200 customers flagged
9 customers have a risk score of 3



,customer_id,risk_score,days_inactive,orders_prev_30d,orders_last_30d,total_spend,sig_inactive,sig_freq_drop,sig_high_value_quiet
102,C1102,3,41,9,4,4971,True,True,True
68,C1068,3,47,1,0,4885,True,True,True
97,C1097,3,31,3,0,4823,True,True,True
158,C1158,3,122,1,0,4329,True,True,True
84,C1084,3,39,3,1,4272,True,True,True
18,C1018,3,66,2,1,4114,True,True,True
57,C1057,3,30,2,1,4072,True,True,True
131,C1131,3,52,4,2,4064,True,True,True
162,C1162,3,65,8,4,3674,True,True,True
35,C1035,2,45,8,10,4962,True,False,True
